In [19]:
# Cell 1: Imports and reproducibility

from pathlib import Path
import os
import random
import json

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Seed:", SEED)

TensorFlow version: 2.16.1
Seed: 42


In [20]:
# ==============================
# Cell 2: Paths
# ==============================

PROJECT_ROOT = (
    Path.cwd().resolve().parents[0]
    if "notebooks" in str(Path.cwd())
    else Path.cwd()
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "models"
LOG_DIR = OUTPUT_DIR / "logs"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_DIR:", MODEL_DIR)

PROJECT_ROOT: /home/harman/projects/Sachin_Epilepsy_Revised
MODEL_DIR: /home/harman/projects/Sachin_Epilepsy_Revised/outputs/models


In [21]:
# ==============================
# Cell 3: Hyperparameter configuration
# ==============================

MODEL_CONFIG = {
    "model_name": "FA-CNNViT",
    "random_seed": 42,

    # Input
    "window_samples": 1024,
    "n_channels_chbmit": 18,
    "n_channels_turkish": 36,

    # Frequency-aware representation
    "freq_filters": 32,
    "multi_scale_kernels": [7, 15, 31],
    "representation_dropout": 0.10,

    # CNN local encoder
    "cnn_filters_1": 32,
    "cnn_filters_2": 64,
    "cnn_kernel_1": (3, 7),
    "cnn_kernel_2": (3, 5),
    "cnn_dropout": 0.20,

    # ViT global encoder
    "patch_size": 16,
    "embed_dim": 128,
    "num_heads": 4,
    "transformer_depth": 2,
    "mlp_dim": 256,
    "transformer_dropout": 0.10,

    # Fusion
    "fusion_dim": 128,
    "fusion_dropout": 0.25,

    # Classifier
    "classifier_hidden": 128,
    "classifier_dropout": 0.30,

    # Uncertainty inference
    "mc_dropout_passes": 20,

    # Optimization
    "optimizer": "Adam",
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "loss_function": "Asymmetric Focal Loss",
    "batch_size": 32,
    "epochs": 50,
    "early_stopping_patience": 10,
}

# Convert to DataFrame for table display
config_df = pd.DataFrame(
    [{"Parameter": k, "Value": str(v)} for k, v in MODEL_CONFIG.items()]
)

config_df.head(20)

,Parameter,Value
0,model_name,FA-CNNViT
1,random_seed,42
2,window_samples,1024
3,n_channels_chbmit,18
4,n_channels_turkish,36
5,freq_filters,32
6,multi_scale_kernels,"[7, 15, 31]"
7,representation_dropout,0.1
8,cnn_filters_1,32
9,cnn_filters_2,64


In [22]:
# ==============================
# Cell 4: Save configuration
# ==============================

import json

# Define file paths
config_csv = TABLE_DIR / "fa_cnnvit_hyperparameters.csv"
config_json = LOG_DIR / "fa_cnnvit_hyperparameters.json"

# Save configuration table (CSV)
config_df.to_csv(config_csv, index=False)

# Save configuration dictionary (JSON)
with open(config_json, "w", encoding="utf-8") as f:
    json.dump(MODEL_CONFIG, f, indent=2)

# Confirmation output
print("Configuration files saved successfully:")
print(f" - CSV : {config_csv}")
print(f" - JSON: {config_json}")

Configuration files saved successfully:
 - CSV : /home/harman/projects/Sachin_Epilepsy_Revised/outputs/tables/fa_cnnvit_hyperparameters.csv
 - JSON: /home/harman/projects/Sachin_Epilepsy_Revised/outputs/logs/fa_cnnvit_hyperparameters.json


In [23]:
# ==============================
# Cell 5: Utility Blocks
# ==============================

from tensorflow.keras import layers


# --------------------------------------------------
# Convolution + BatchNorm + Activation (+ Dropout)
# --------------------------------------------------
def conv_bn_act(x, filters, kernel_size, strides=(1, 1),
                dropout_rate=0.0, name_prefix="conv"):
    
    x = layers.Conv2D(
        filters=filters,
        kernel_size=kernel_size,
        strides=strides,
        padding="same",
        use_bias=False,
        name=f"{name_prefix}_conv"
    )(x)

    x = layers.BatchNormalization(name=f"{name_prefix}_bn")(x)
    x = layers.Activation("relu", name=f"{name_prefix}_relu")(x)

    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate, name=f"{name_prefix}_drop")(x)

    return x


# --------------------------------------------------
# MLP Block (used in Transformer)
# --------------------------------------------------
class MLPBlock(layers.Layer):
    def __init__(self, mlp_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.fc1 = layers.Dense(mlp_dim, activation="gelu")
        self.drop1 = layers.Dropout(dropout_rate)
        self.fc2 = layers.Dense(mlp_dim)
        self.drop2 = layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        x = self.fc1(x)
        x = self.drop1(x, training=training)
        x = self.fc2(x)
        x = self.drop2(x, training=training)
        return x


# --------------------------------------------------
# Transformer Encoder Block
# --------------------------------------------------
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, num_heads, mlp_dim,
                 dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)

        # Layer Normalization + Multi-Head Attention
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout_rate
        )
        self.drop1 = layers.Dropout(dropout_rate)

        # Feed Forward Network (MLP)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp1 = layers.Dense(mlp_dim, activation="gelu")
        self.drop2 = layers.Dropout(dropout_rate)
        self.mlp2 = layers.Dense(embed_dim)
        self.drop3 = layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        # Multi-head Self-Attention (Residual Connection)
        x1 = self.norm1(x)
        attn_out = self.attn(x1, x1, training=training)
        attn_out = self.drop1(attn_out, training=training)
        x = x + attn_out

        # Feed Forward Network (Residual Connection)
        x2 = self.norm2(x)
        mlp_out = self.mlp1(x2)
        mlp_out = self.drop2(mlp_out, training=training)
        mlp_out = self.mlp2(mlp_out)
        mlp_out = self.drop3(mlp_out, training=training)
        x = x + mlp_out

        return x

In [24]:
# ==============================================
# Cell 6: Frequency-Aware Multi-Scale Representation
# ==============================================

import tensorflow as tf
from tensorflow.keras import layers


def build_frequency_aware_representation(
    x,
    freq_filters=32,
    multi_scale_kernels=(7, 15, 31),
    dropout_rate=0.10,
    name_prefix="freq"
):
    """
    Frequency-Aware Multi-Scale Feature Extraction Block

    Input:
        x -> shape: (channels, time, 1)

    Output:
        Feature map for downstream CNN and ViT branches
    """

    # --------------------------------------------------
    # 1. Multi-scale temporal convolutions
    # --------------------------------------------------
    multi_scale_feats = []

    for i, k in enumerate(multi_scale_kernels):
        branch = conv_bn_act(
            x,
            filters=freq_filters,
            kernel_size=(1, k),
            dropout_rate=dropout_rate,
            name_prefix=f"{name_prefix}_ms{i+1}"
        )
        multi_scale_feats.append(branch)

    x_ms = layers.Concatenate(name=f"{name_prefix}_concat")(multi_scale_feats)

    # --------------------------------------------------
    # 2. Channel-wise spatial energy proxy
    # --------------------------------------------------
    energy_map = layers.Lambda(
        lambda t: tf.reduce_mean(tf.square(t), axis=2, keepdims=True),
        name=f"{name_prefix}_energy"
    )(x_ms)

    # Tile energy map to match temporal dimension
    x_width = x_ms.shape[2]

    energy_map = layers.Lambda(
        lambda t: tf.repeat(t, repeats=x_width, axis=2),
        name=f"{name_prefix}_energy_tile"
    )(energy_map)

    # Concatenate energy-aware features
    x_fused = layers.Concatenate(name=f"{name_prefix}_energy_concat")(
        [x_ms, energy_map]
    )

    # --------------------------------------------------
    # 3. Band-aware gating (Attention-like mechanism)
    # --------------------------------------------------
    x_proj = conv_bn_act(
        x_fused,
        filters=freq_filters * 2,
        kernel_size=(1, 1),
        dropout_rate=dropout_rate,
        name_prefix=f"{name_prefix}_proj"
    )

    gate = layers.GlobalAveragePooling2D(name=f"{name_prefix}_gap")(x_proj)

    gate = layers.Dense(
        freq_filters * 2,
        activation="sigmoid",
        name=f"{name_prefix}_gate"
    )(gate)

    gate = layers.Reshape(
        (1, 1, freq_filters * 2),
        name=f"{name_prefix}_gate_reshape"
    )(gate)

    # Apply gating
    x_out = layers.Multiply(name=f"{name_prefix}_gated")([x_proj, gate])

    return x_out

In [25]:
# ==============================================
# Cell 7: CNN Local Encoder
# ==============================================

from tensorflow.keras import layers


def build_cnn_local_encoder(
    x,
    filters_1=32,
    filters_2=64,
    kernel_1=(3, 7),
    kernel_2=(3, 5),
    dropout_rate=0.20,
    embedding_dim=128,
    name_prefix="cnn_local"
):
    """
    CNN-based Local Feature Encoder

    Purpose:
        Captures local spatial-temporal patterns from EEG signals

    Input:
        x -> Feature map from frequency-aware block

    Output:
        Dense embedding vector for fusion with ViT branch
    """

    # --------------------------------------------------
    # Block 1
    # --------------------------------------------------
    x = conv_bn_act(
        x,
        filters=filters_1,
        kernel_size=kernel_1,
        dropout_rate=dropout_rate,
        name_prefix=f"{name_prefix}_block1"
    )

    x = layers.MaxPooling2D(
        pool_size=(1, 2),
        name=f"{name_prefix}_pool1"
    )(x)

    # --------------------------------------------------
    # Block 2
    # --------------------------------------------------
    x = conv_bn_act(
        x,
        filters=filters_2,
        kernel_size=kernel_2,
        dropout_rate=dropout_rate,
        name_prefix=f"{name_prefix}_block2"
    )

    x = layers.MaxPooling2D(
        pool_size=(1, 2),
        name=f"{name_prefix}_pool2"
    )(x)

    # --------------------------------------------------
    # Global Feature Aggregation
    # --------------------------------------------------
    x = layers.GlobalAveragePooling2D(
        name=f"{name_prefix}_gap"
    )(x)

    # --------------------------------------------------
    # Embedding Projection
    # --------------------------------------------------
    x = layers.Dense(
        embedding_dim,
        activation="relu",
        name=f"{name_prefix}_embedding"
    )(x)

    return x

In [26]:
# ==============================================
# Corrected Cell: ViT Global Encoder (Keras 3 Compatible)
# ==============================================

import tensorflow as tf
from tensorflow.keras import layers


# --------------------------------------------------
# Patch + Position Embedding (Keras 3 Safe)
# --------------------------------------------------
class PatchAndPositionEmbedding(layers.Layer):
    def __init__(self, patch_size=16, embed_dim=128, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size
        self.embed_dim = embed_dim

        self.proj = layers.Conv2D(
            filters=self.embed_dim,
            kernel_size=(1, self.patch_size),
            strides=(1, self.patch_size),
            padding="valid",
            name=f"{self.name}_proj"
        )

        self.pos_embedding = None
        self.n_tokens = None

    def build(self, input_shape):
        # Build projection layer
        self.proj.build(input_shape)

        # Compute output shape after projection
        proj_shape = self.proj.compute_output_shape(input_shape)
        h_tokens = int(proj_shape[1])
        w_tokens = int(proj_shape[2])

        self.n_tokens = h_tokens * w_tokens

        # Create positional embedding (FIXED: not in call)
        self.pos_embedding = self.add_weight(
            name="pos_embedding",
            shape=(1, self.n_tokens, self.embed_dim),
            initializer="random_normal",
            trainable=True
        )

        super().build(input_shape)

    def call(self, x):
        x = self.proj(x)  # (B, H, W, C)
        batch_size = tf.shape(x)[0]

        x = tf.reshape(x, (batch_size, self.n_tokens, self.embed_dim))
        return x + self.pos_embedding

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.n_tokens, self.embed_dim)


# --------------------------------------------------
# ViT Global Encoder
# --------------------------------------------------
def build_vit_global_encoder(
    x,
    patch_size=16,
    embed_dim=128,
    num_heads=4,
    transformer_depth=2,
    mlp_dim=256,
    dropout_rate=0.10,
    name_prefix="vit_global"
):

    # Patch embedding + positional encoding
    x = PatchAndPositionEmbedding(
        patch_size=patch_size,
        embed_dim=embed_dim,
        name=f"{name_prefix}_patch_embed"
    )(x)

    # Transformer encoder blocks
    for i in range(transformer_depth):
        x = TransformerEncoder(
            embed_dim=embed_dim,
            num_heads=num_heads,
            mlp_dim=mlp_dim,
            dropout_rate=dropout_rate,
            name=f"{name_prefix}_block{i+1}"
        )(x)

    # Final normalization and pooling
    x = layers.LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_norm")(x)
    x = layers.GlobalAveragePooling1D(name=f"{name_prefix}_gap")(x)

    # Projection
    x = layers.Dense(
        embed_dim,
        activation="relu",
        name=f"{name_prefix}_embedding"
    )(x)

    return x

In [27]:
# ==============================================
# Cell 9: Adaptive Multi-Scale Fusion Block
# ==============================================

from tensorflow.keras import layers


def build_adaptive_fusion(
    local_feat,
    global_feat,
    fusion_dim=128,
    dropout_rate=0.25,
    name_prefix="fusion"
):
    """
    Adaptive Fusion of CNN (local) and ViT (global) features

    Purpose:
        Learns dynamic weighting between local and global representations

    Inputs:
        local_feat  -> CNN embedding
        global_feat -> ViT embedding

    Output:
        Fused representation for classification
    """

    # --------------------------------------------------
    # 1. Projection to common latent space
    # --------------------------------------------------
    local_proj = layers.Dense(
        fusion_dim,
        activation="relu",
        name=f"{name_prefix}_local_proj"
    )(local_feat)

    global_proj = layers.Dense(
        fusion_dim,
        activation="relu",
        name=f"{name_prefix}_global_proj"
    )(global_feat)

    # --------------------------------------------------
    # 2. Fusion context (joint representation)
    # --------------------------------------------------
    fusion_context = layers.Concatenate(
        name=f"{name_prefix}_concat"
    )([local_proj, global_proj])

    # --------------------------------------------------
    # 3. Gating mechanism (adaptive weighting)
    # --------------------------------------------------
    gate = layers.Dense(
        fusion_dim,
        activation="sigmoid",
        name=f"{name_prefix}_gate"
    )(fusion_context)

    # Local contribution
    gated_local = layers.Multiply(
        name=f"{name_prefix}_gated_local"
    )([gate, local_proj])

    # Global contribution (complementary gate)
    inv_gate = layers.Lambda(
        lambda t: 1.0 - t,
        name=f"{name_prefix}_inv_gate"
    )(gate)

    gated_global = layers.Multiply(
        name=f"{name_prefix}_gated_global"
    )([inv_gate, global_proj])

    # --------------------------------------------------
    # 4. Fusion aggregation
    # --------------------------------------------------
    fused = layers.Add(name=f"{name_prefix}_add")(
        [gated_local, gated_global]
    )

    # Regularization
    fused = layers.Dropout(
        dropout_rate,
        name=f"{name_prefix}_drop"
    )(fused)

    # Final embedding
    fused = layers.Dense(
        fusion_dim,
        activation="relu",
        name=f"{name_prefix}_embedding"
    )(fused)

    return fused

In [28]:
# ==============================================
# Cell 10: Uncertainty-Aware Classifier Head
# ==============================================

from tensorflow.keras import layers


def build_uncertainty_classifier(
    fused_embedding,
    hidden_dim=128,
    dropout_rate=0.30,
    name_prefix="uncertainty_head"
):
    """
    Uncertainty-Aware Classification Head

    Purpose:
        Performs final classification with implicit uncertainty estimation
        via dropout-based stochastic regularization (MC Dropout compatible)

    Input:
        fused_embedding -> Output from adaptive fusion block

    Output:
        prob -> Seizure probability (binary classification)
    """

    # --------------------------------------------------
    # Fully Connected Block 1
    # --------------------------------------------------
    x = layers.Dense(
        hidden_dim,
        activation="relu",
        name=f"{name_prefix}_fc1"
    )(fused_embedding)

    x = layers.Dropout(
        dropout_rate,
        name=f"{name_prefix}_drop1"
    )(x)

    # --------------------------------------------------
    # Fully Connected Block 2
    # --------------------------------------------------
    x = layers.Dense(
        hidden_dim // 2,
        activation="relu",
        name=f"{name_prefix}_fc2"
    )(x)

    x = layers.Dropout(
        dropout_rate,
        name=f"{name_prefix}_drop2"
    )(x)

    # --------------------------------------------------
    # Output Layer (Binary Classification)
    # --------------------------------------------------
    prob = layers.Dense(
        1,
        activation="sigmoid",
        name=f"{name_prefix}_prob"
    )(x)

    return prob

In [29]:
# ==============================================
# Cell 11: Full FA-CNNViT Model Builder
# ==============================================

from tensorflow import keras
from tensorflow.keras import layers


def build_fa_cnnvit(
    n_channels,
    window_samples,
    config,
    model_name="FA-CNNViT"
):
    """
    Full FA-CNNViT Architecture

    Pipeline:
        Input → Frequency-Aware Representation →
        [CNN Local Branch + ViT Global Branch] →
        Adaptive Fusion → Uncertainty-Aware Classifier

    Inputs:
        n_channels     -> Number of EEG channels
        window_samples -> Time window length
        config         -> Hyperparameter dictionary

    Output:
        Keras Model
    """

    # --------------------------------------------------
    # Input Layer
    # --------------------------------------------------
    inputs = layers.Input(
        shape=(n_channels, window_samples, 1),
        name="eeg_input"
    )

    # --------------------------------------------------
    # Frequency-Aware Representation
    # --------------------------------------------------
    rep = build_frequency_aware_representation(
        inputs,
        freq_filters=config["freq_filters"],
        multi_scale_kernels=config["multi_scale_kernels"],
        dropout_rate=config["representation_dropout"],
        name_prefix="freq_repr"
    )

    # --------------------------------------------------
    # CNN Local Branch
    # --------------------------------------------------
    local_feat = build_cnn_local_encoder(
        rep,
        filters_1=config["cnn_filters_1"],
        filters_2=config["cnn_filters_2"],
        kernel_1=config["cnn_kernel_1"],
        kernel_2=config["cnn_kernel_2"],
        dropout_rate=config["cnn_dropout"],
        embedding_dim=config["fusion_dim"],
        name_prefix="cnn_local"
    )

    # --------------------------------------------------
    # ViT Global Branch
    # --------------------------------------------------
    global_feat = build_vit_global_encoder(
        rep,
        patch_size=config["patch_size"],
        embed_dim=config["embed_dim"],
        num_heads=config["num_heads"],
        transformer_depth=config["transformer_depth"],
        mlp_dim=config["mlp_dim"],
        dropout_rate=config["transformer_dropout"],
        name_prefix="vit_global"
    )

    # --------------------------------------------------
    # Adaptive Fusion
    # --------------------------------------------------
    fused = build_adaptive_fusion(
        local_feat,
        global_feat,
        fusion_dim=config["fusion_dim"],
        dropout_rate=config["fusion_dropout"],
        name_prefix="adaptive_fusion"
    )

    # --------------------------------------------------
    # Uncertainty-Aware Classifier
    # --------------------------------------------------
    outputs = build_uncertainty_classifier(
        fused,
        hidden_dim=config["classifier_hidden"],
        dropout_rate=config["classifier_dropout"],
        name_prefix="uncertainty_head"
    )

    # --------------------------------------------------
    # Model Assembly
    # --------------------------------------------------
    model = keras.Model(
        inputs=inputs,
        outputs=outputs,
        name=model_name
    )

    return model

In [30]:
# ==============================================
# Cell 12: Build CHB-MIT and Turkish Variants
# ==============================================

# Instantiate model for CHB-MIT dataset
model_chbmit = build_fa_cnnvit(
    n_channels=MODEL_CONFIG["n_channels_chbmit"],
    window_samples=MODEL_CONFIG["window_samples"],
    config=MODEL_CONFIG,
    model_name="FA-CNNViT_CHBMIT"
)

# Instantiate model for Turkish EEG dataset
model_turkish = build_fa_cnnvit(
    n_channels=MODEL_CONFIG["n_channels_turkish"],
    window_samples=MODEL_CONFIG["window_samples"],
    config=MODEL_CONFIG,
    model_name="FA-CNNViT_Turkish"
)

# --------------------------------------------------
# Verification Output
# --------------------------------------------------
print("Models successfully instantiated:")
print(f" - {model_chbmit.name}")
print(f" - {model_turkish.name}")

Models successfully instantiated:
 - FA-CNNViT_CHBMIT
 - FA-CNNViT_Turkish


In [31]:
# ==============================================
# Cell 13: Model Summaries
# ==============================================

# --------------------------------------------------
# CHB-MIT Model Summary
# --------------------------------------------------
print("\n=== CHB-MIT Model Summary ===")
model_chbmit.summary()

# --------------------------------------------------
# Turkish Model Summary
# --------------------------------------------------
print("\n=== Turkish Model Summary ===")
model_turkish.summary()


=== CHB-MIT Model Summary ===


Model: "FA-CNNViT_CHBMIT"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ eeg_input           │ (None, 18, 1024,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms1_conv  │ (None, 18, 1024,  │        224 │ eeg_input[0][0]   │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms2_conv  │ (None, 18, 1024,  │        480 │ eeg_input[0][0]   │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms3_conv  │ (None, 18, 1024,  │        992 │ eeg_input[0][0]   │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms1_bn    │ (None, 18, 1024,  │        128 │ freq_repr_ms1_co… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms2_bn    │ (None, 18, 1024,  │        128 │ freq_repr_ms2_co… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms3_bn    │ (None, 18, 1024,  │        128 │ freq_repr_ms3_co… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms1_relu  │ (None, 18, 1024,  │          0 │ freq_repr_ms1_bn… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms2_relu  │ (None, 18, 1024,  │          0 │ freq_repr_ms2_bn… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms3_relu  │ (None, 18, 1024,  │          0 │ freq_repr_ms3_bn… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms1_drop  │ (None, 18, 1024,  │          0 │ freq_repr_ms1_re… │
│ (Dropout)           │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms2_drop  │ (None, 18, 1024,  │          0 │ freq_repr_ms2_re… │
│ (Dropout)           │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms3_drop  │ (None, 18, 1024,  │          0 │ freq_repr_ms3_re… │
│ (Dropout)           │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_concat    │ (None, 18, 1024,  │          0 │ freq_repr_ms1_dr… │
│ (Concatenate)       │ 96)               │            │ freq_repr_ms2_dr… │
│                     │                   │            │ freq_repr_ms3_dr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_energy    │ (None, 18, 1, 96) │          0 │ freq_repr_concat… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_energy_t… │ (None, 18, 1024,  │          0 │ freq_repr_energy… │
│ (Lambda)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 768,865 (2.93 MB)

 Trainable params: 768,353 (2.93 MB)

 Non-trainable params: 512 (2.00 KB)

... (2 outputs left out)

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ eeg_input           │ (None, 36, 1024,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms1_conv  │ (None, 36, 1024,  │        224 │ eeg_input[0][0]   │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms2_conv  │ (None, 36, 1024,  │        480 │ eeg_input[0][0]   │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms3_conv  │ (None, 36, 1024,  │        992 │ eeg_input[0][0]   │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms1_bn    │ (None, 36, 1024,  │        128 │ freq_repr_ms1_co… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms2_bn    │ (None, 36, 1024,  │        128 │ freq_repr_ms2_co… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms3_bn    │ (None, 36, 1024,  │        128 │ freq_repr_ms3_co… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms1_relu  │ (None, 36, 1024,  │          0 │ freq_repr_ms1_bn… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms2_relu  │ (None, 36, 1024,  │          0 │ freq_repr_ms2_bn… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms3_relu  │ (None, 36, 1024,  │          0 │ freq_repr_ms3_bn… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms1_drop  │ (None, 36, 1024,  │          0 │ freq_repr_ms1_re… │
│ (Dropout)           │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms2_drop  │ (None, 36, 1024,  │          0 │ freq_repr_ms2_re… │
│ (Dropout)           │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_ms3_drop  │ (None, 36, 1024,  │          0 │ freq_repr_ms3_re… │
│ (Dropout)           │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_concat    │ (None, 36, 1024,  │          0 │ freq_repr_ms1_dr… │
│ (Concatenate)       │ 96)               │            │ freq_repr_ms2_dr… │
│                     │                   │            │ freq_repr_ms3_dr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_energy    │ (None, 36, 1, 96) │          0 │ freq_repr_concat… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_repr_energy_t… │ (None, 36, 1024,  │          0 │ freq_repr_energy… │
│ (Lambda)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 916,321 (3.50 MB)

 Trainable params: 915,809 (3.49 MB)

 Non-trainable params: 512 (2.00 KB)

In [32]:
# ==============================================
# Cell 14: Dummy Forward Pass
# ==============================================

import numpy as np

# --------------------------------------------------
# Generate dummy inputs
# --------------------------------------------------
dummy_chb = np.random.randn(
    2,
    MODEL_CONFIG["n_channels_chbmit"],
    MODEL_CONFIG["window_samples"],
    1
).astype(np.float32)

dummy_turk = np.random.randn(
    2,
    MODEL_CONFIG["n_channels_turkish"],
    MODEL_CONFIG["window_samples"],
    1
).astype(np.float32)

# --------------------------------------------------
# Forward pass (inference mode)
# --------------------------------------------------
out_chb = model_chbmit(dummy_chb, training=False)
out_turk = model_turkish(dummy_turk, training=False)

# --------------------------------------------------
# Output verification
# --------------------------------------------------
print("=== Forward Pass Verification ===")

print("\nCHB-MIT output shape:", out_chb.shape)
print("Turkish output shape:", out_turk.shape)

print("\nCHB-MIT sample output:", out_chb.numpy().ravel()[:4])
print("Turkish sample output:", out_turk.numpy().ravel()[:4])

=== Forward Pass Verification ===

CHB-MIT output shape: (2, 1)
Turkish output shape: (2, 1)

CHB-MIT sample output: [0.49125296 0.49190602]
Turkish sample output: [0.46434358 0.4642304 ]


In [33]:
# ==============================================
# Cell 15: Save Model Configs and Parameter Counts
# ==============================================

import pandas as pd

# --------------------------------------------------
# Create parameter summary table
# --------------------------------------------------
param_summary_df = pd.DataFrame([
    {
        "model_variant": "CHB-MIT",
        "n_channels": MODEL_CONFIG["n_channels_chbmit"],
        "window_samples": MODEL_CONFIG["window_samples"],
        "trainable_params": model_chbmit.count_params()
    },
    {
        "model_variant": "Turkish",
        "n_channels": MODEL_CONFIG["n_channels_turkish"],
        "window_samples": MODEL_CONFIG["window_samples"],
        "trainable_params": model_turkish.count_params()
    }
])

# --------------------------------------------------
# Save to CSV
# --------------------------------------------------
param_csv = TABLE_DIR / "fa_cnnvit_parameter_summary.csv"
param_summary_df.to_csv(param_csv, index=False)

# --------------------------------------------------
# Confirmation + Display
# --------------------------------------------------
print(f"Parameter summary saved at: {param_csv}")

param_summary_df

Parameter summary saved at: /home/harman/projects/Sachin_Epilepsy_Revised/outputs/tables/fa_cnnvit_parameter_summary.csv


,model_variant,n_channels,window_samples,trainable_params
0,CHB-MIT,18,1024,768865
1,Turkish,36,1024,916321


In [34]:
import shutil
print(shutil.which("dot"))

/usr/bin/dot


In [35]:
# ==============================================
# Cell 16: Optional Model Diagrams
# ==============================================

from tensorflow import keras

# --------------------------------------------------
# Plot and save model architecture (CHB-MIT)
# --------------------------------------------------
try:
    chbmit_plot_path = MODEL_DIR / "fa_cnnvit_chbmit_architecture.png"

    keras.utils.plot_model(
        model_chbmit,
        to_file=str(chbmit_plot_path),
        show_shapes=True,
        show_dtype=False,
        expand_nested=False
    )

    print(f"Model diagram saved at: {chbmit_plot_path}")

except Exception as e:
    print("plot_model skipped due to error:")
    print(e)

Model diagram saved at: /home/harman/projects/Sachin_Epilepsy_Revised/outputs/models/fa_cnnvit_chbmit_architecture.png


In [36]:
!pip install graphviz pydot